In [20]:
import os
import re
import numpy as np
import pandas as pd
import gspread
from google.colab import auth, drive
from google.auth import default

In [21]:
# Montagem do Google Drive
drive.mount("/content/drive")

# Autenticacao e leitura da planilha Google Sheets
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1x-lDeKhbvO-ekjFNFDRsa3QFO-0gfC_2oYPHOKwDqEo/edit?gid=1784516557#gid=1784516557"
sh = gc.open_by_url(SPREADSHEET_URL)
worksheet = sh.get_worksheet_by_id(1784516557)
rows = worksheet.get_all_values()

df_raw = pd.DataFrame(rows[1:], columns=rows[0])

# Validacao rapida do Bloco 1
print(f"Shape bruto: {df_raw.shape}")
print("\nColunas carregadas:", list(df_raw.columns))
display(df_raw.head(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Shape bruto: (1465, 16)

Colunas carregadas: ['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B0912WJ87V,Reffair AX30 [MAX] Portable Air Purifier for C...,Car&Motorbike|CarAccessories|InteriorAccessori...,"₹2,339","₹4,000",42%,3.8,"1,118",[NEW LAUNCH - INNOVATIVE DESIGN] EFFICIENT PUR...,"AG6W5HESRSDLBX3NCYOOUGFOWERA,AED4U5CF37HVSYPMD...","Arivazhagan,Ajit,swanand pandit,Venkatesh S.,S...","R3TOOFPX256D59,R1PR50BDQOEIIO,R392FI4QWXWOX5,R...","Liked it,User friendly and effective,Not as pe...","As of now its working fine, but don't expect m...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Reffair-AX30-MAX-Interna...
1,B097C564GC,"rts [2 Pack] Mini USB C Type C Adapter Plug, T...",Computers&Accessories|Accessories&Peripherals|...,₹294,"₹4,999",94%,4.3,"4,426",【WIDER COMPATIBILITY】This USB C Female to USB ...,"AGO5SRT3ESLNL5WTRQFD5BSIYPQA,AGGNODUMRADB3PQQT...","Naga Sai Gadagottu,Neha Sharma,Amazon Customer...","R3CUNCZTU43JPP,RSO46BN8S4OSU,R2UD5D7T4DZRE5,R2...","Tiny helping hand,Good product,As a whole good...",This eliminates the need to buy chargers for t...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/rts-Adapter-Charging-Con...
2,B094DQWV9B,Kanget [2 Pack] Type C Female to USB A Male Ch...,Computers&Accessories|Accessories&Peripherals|...,₹149,₹399,63%,4.0,"1,54",👍【USB C TO USB ADAPTER】-This is a USB C FEMALE...,"AFA6YM4NTFRGHHKYN5U7HUYEVSUA,AEB3EVUTQCHIE3PGI...","Abhi_K,Amazon Customer,Swap,shivam,Abhinav Sin...","R1QIWMR6C3F3U0,R1MSGOZTOMZE4B,R20OZCEE82GU0W,R...","Good and does it’s work,Nice product working a...",Using it to connect my type C Plantronic headp...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Kanget-Female-Adapter-St...


In [22]:
def clean_currency(value):
    if pd.isna(value):
        return np.nan
    cleaned = re.sub(r"[^\d.]", "", str(value))
    return float(cleaned) if cleaned else np.nan


def clean_percentage(value):
    if pd.isna(value):
        return np.nan
    cleaned = re.sub(r"[^\d.]", "", str(value))
    return round(float(cleaned) / 100.0, 4) if cleaned else np.nan


def clean_rating(value):
    if pd.isna(value):
        return np.nan
    match = re.search(r"\d+(\.\d+)?", str(value))
    if not match:
        return np.nan
    val = float(match.group(0))
    return val if 1.0 <= val <= 5.0 else np.nan


def clean_int(value):
    if pd.isna(value):
        return np.nan
    cleaned = re.sub(r"[^\d]", "", str(value))
    return int(cleaned) if cleaned else np.nan


def get_discount_range(pct):
    if pd.isna(pct):
        return np.nan
    pct_100 = pct * 100.0
    if pct_100 <= 10:
        return "0% - 10%"
    elif pct_100 <= 20:
        return "10% - 20%"
    elif pct_100 <= 30:
        return "20% - 30%"
    elif pct_100 <= 40:
        return "30% - 40%"
    elif pct_100 <= 50:
        return "40% - 50%"
    elif pct_100 <= 60:
        return "50% - 60%"
    elif pct_100 <= 70:
        return "60% - 70%"
    elif pct_100 <= 80:
        return "70% - 80%"
    elif pct_100 <= 90:
        return "80% - 90%"
    else:
        return "90% - 100%"


def clean_review_text(text):
    if pd.isna(text):
        return np.nan
    text_no_urls = re.sub(r"https?://\S+", "", str(text))
    cleaned = re.sub(r"\s+", " ", text_no_urls).strip(" ,.-_")
    return cleaned if cleaned else np.nan


def classify_sentiment(title, content, product_rating):
    combined = f"{'' if pd.isna(title) else title} {'' if pd.isna(content) else content}".lower().strip()
    if not combined and pd.isna(product_rating):
        return np.nan

    pos_words = {
        "good", "great", "excellent", "awesome", "best", "nice", "worth",
        "amazing", "perfect", "satisfied", "love", "super", "durable",
        "fast", "value", "reliable", "happy", "smooth", "sturdy", "fine"
    }
    neg_words = {
        "bad", "poor", "worst", "waste", "broken", "issue", "problem",
        "defective", "stopped", "slow", "disappointed", "useless", "cheap",
        "not working", "return", "damaged", "fake", "heating", "terrible"
    }

    tokens = set(re.findall(r"\b\w+\b", combined))
    pos_score = len(tokens & pos_words)
    neg_score = len(tokens & neg_words)

    if "not working" in combined or "not good" in combined or "not durable" in combined:
        neg_score += 2

    if pos_score > neg_score:
        return "Positive"
    elif neg_score > pos_score:
        return "Negative"
    else:
        if pd.notna(product_rating):
            if product_rating >= 4.0:
                return "Positive"
            elif product_rating <= 3.2:
                return "Negative"
        return "Neutral" if combined else np.nan


def split_reviews_row(row):
    pid = row["product_id"]
    p_rating = row["rating"]

    r_ids = [x.strip() for x in str(row["review_id"]).split(",")] if pd.notna(row["review_id"]) else []
    u_ids = [x.strip() for x in str(row["user_id"]).split(",")] if pd.notna(row["user_id"]) else []
    u_names = [x.strip() for x in str(row["user_name"]).split(",")] if pd.notna(row["user_name"]) else []
    r_titles = [x.strip() for x in str(row["review_title"]).split(",")] if pd.notna(row["review_title"]) else []

    raw_content = str(row["review_content"]) if pd.notna(row["review_content"]) else ""
    raw_content_clean = re.sub(r"https?://\S+", "", raw_content)
    r_contents = [x.strip() for x in raw_content_clean.split(",") if x.strip()]

    n_reviews = max(len(r_ids), len(u_ids), 1) if any([r_ids, u_ids, r_titles, r_contents]) else 0
    records = []

    for i in range(n_reviews):
        rev_id = r_ids[i] if i < len(r_ids) and r_ids[i] else np.nan
        usr_id = u_ids[i] if i < len(u_ids) and u_ids[i] else np.nan
        usr_name = u_names[i] if i < len(u_names) and u_names[i] else np.nan
        rev_title = clean_review_text(r_titles[i]) if i < len(r_titles) else np.nan

        if i < len(r_contents):
            if i == n_reviews - 1 and len(r_contents) > n_reviews:
                rev_content = clean_review_text(", ".join(r_contents[i:]))
            else:
                rev_content = clean_review_text(r_contents[i])
        else:
            rev_content = np.nan

        word_num = len(rev_content.split()) if pd.notna(rev_content) else np.nan
        rev_res = classify_sentiment(rev_title, rev_content, p_rating)

        records.append({
            "product_id": pid,
            "review_id": rev_id,
            "user_id": usr_id,
            "user_name": usr_name,
            "review_title": rev_title,
            "review_content": rev_content,
            "word_number": word_num,
            "review_result": rev_res
        })

    return records


# Validacao rapida do Bloco 2
print("Funcoes carregadas com sucesso.")
print("Teste get_discount_range(0.43):", get_discount_range(0.43))

Funcoes carregadas com sucesso.
Teste get_discount_range(0.43): 40% - 50%


In [23]:
df_clean = df_raw.copy()

# 1. Remove colunas que nao serao utilizadas
cols_to_drop = ["img_link", "product_link", "about_product"]
df_clean.drop(columns=[c for c in cols_to_drop if c in df_clean.columns], inplace=True)

# 2. Padroniza strings vazias e espacos extras como nulo
for col in df_clean.columns:
    df_clean[col] = df_clean[col].astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})

# 3. Limpeza dos campos numericos
df_clean["actual_price_num"] = df_clean["actual_price"].apply(clean_currency)
df_clean["discounted_price_num"] = df_clean["discounted_price"].apply(clean_currency)
df_clean["discount_percentage_num"] = df_clean["discount_percentage"].apply(clean_percentage)
df_clean["rating_num"] = df_clean["rating"].apply(clean_rating)
df_clean["rating_count_num"] = df_clean["rating_count"].apply(clean_int)

# 4. Deduplicacao mantendo o registro com maior rating_count
df_clean = (
    df_clean.sort_values(by=["product_id", "rating_count_num"], ascending=[True, False], na_position="last")
    .drop_duplicates(subset=["product_id"], keep="first")
    .reset_index(drop=True)
)

# Validacao rapida do Bloco 3
print(f"Shape apos limpeza e deduplicacao: {df_clean.shape}")
print("\nContagem de nulos apos tratamento:")
print(df_clean[["product_id", "actual_price_num", "discounted_price_num", "discount_percentage_num", "rating_num", "rating_count_num"]].isna().sum())
display(df_clean.head(3))

Shape apos limpeza e deduplicacao: (1351, 18)

Contagem de nulos apos tratamento:
product_id                 0
actual_price_num           0
discounted_price_num       0
discount_percentage_num    0
rating_num                 1
rating_count_num           2
dtype: int64


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,user_id,user_name,review_id,review_title,review_content,actual_price_num,discounted_price_num,discount_percentage_num,rating_num,rating_count_num
0,B002PD61Y4,D-Link DWA-131 300 Mbps Wireless Nano USB Adap...,Computers&Accessories|NetworkingDevices|Networ...,₹507,"₹1,208",58%,4.1,"8,131","AGA2PZGWMQIRA46VYOTICFE7KCBA,AHI2QJ4CLTCQWACDI...","nilesh,EAGLE,Manoj KNS,Titus P.,Paras singla,a...","R2EJIN3N3L3XKI,R2JMJ8QNG66LV4,R3B46JNPC2T4E7,R...","good tool to use for,Brand is always good,Over...",good quality tool from d linkWiFi signal is go...,1208.0,507.0,0.58,4.1,8131.0
1,B002SZEOLG,TP-Link Nano USB WiFi Dongle 150Mbps High Gain...,Computers&Accessories|NetworkingDevices|Networ...,₹749,"₹1,339",44%,4.2,"1,79,692","AGV3IEFANZCKECFGUM42MRH5FNOA,AEBO7NWCNXKT4AESA...","Azhar JuMan,Anirudh Sood,Hari Krishnan PS,Akas...","R1LW6NWSVTVZ2H,R3VR5WFKUS15C5,R2F6GC79OYWUKQ,R...",Works on linux for me. Get the model with ante...,I use this to connect an old PC to internet. I...,1339.0,749.0,0.44,4.2,179692.0
2,B003B00484,Duracell Plus AAA Rechargeable Batteries (750 ...,Electronics|GeneralPurposeBatteries&BatteryCha...,₹399,₹499,20%,4.3,"27,201","AG2ICOYPSOV5SGBKFEYHGKCNK7PA,AGJ3OQ4X262D3MAQZ...","T N Sivaji,Akku,V,Meet,MOHAMMED,Niranjan koyri...","R5L3FAFS6JXJF,R1VTQ25LXQX5UD,R6RJYAZUM5240,R1S...","Works Good,Perfect replacement cell for trimme...","Works good,Bought it to replace my Phillips QT...",499.0,399.0,0.20,4.3,27201.0


In [ ]:
# 1. Construcao de dim_products
dim_products = pd.DataFrame()
dim_products["product_id"] = df_clean["product_id"]
dim_products["product_name"] = df_clean["product_name"]
dim_products["category_full"] = df_clean["category"]

# Divisao da arvore de categorias em ate 5 niveis (nulo quando ausente)
cat_splits = dim_products["category_full"].str.split("|", expand=True)
for lvl in range(5):
    col_name = f"category_l{lvl + 1}"
    if lvl in cat_splits.columns:
        dim_products[col_name] = cat_splits[lvl].str.strip().replace({"": np.nan, "nan": np.nan})
    else:
        dim_products[col_name] = np.nan

# Metricas de preco, desconto e faixas
dim_products["actual_price"] = df_clean["actual_price_num"]
dim_products["discounted_price"] = df_clean["discounted_price_num"]
dim_products["discount_amount"] = (dim_products["actual_price"] - dim_products["discounted_price"]).round(2)
dim_products["discount_percentage"] = df_clean["discount_percentage_num"]
dim_products["discount_range"] = dim_products["discount_percentage"].apply(get_discount_range)

# Avaliacoes e niveis de engajamento por quartil
dim_products["rating"] = df_clean["rating_num"]
dim_products["rating_count"] = df_clean["rating_count_num"].astype("Int64")

valid_counts = dim_products["rating_count"].dropna()
q1 = valid_counts.quantile(0.25)
q3 = valid_counts.quantile(0.75)

def get_engagement(val):
    if pd.isna(val):
        return np.nan
    if val <= q1:
        return "Low"
    elif val <= q3:
        return "Medium"
    return "High"

dim_products["engagement_level"] = dim_products["rating_count"].apply(get_engagement)

# 2. Construcao de fact_reviews
df_reviews_input = df_clean[["product_id", "review_id", "user_id", "user_name", "review_title", "review_content", "rating_num"]].rename(
    columns={"rating_num": "rating"}
)

all_reviews = []
for _, row in df_reviews_input.iterrows():
    all_reviews.extend(split_reviews_row(row))

fact_reviews = pd.DataFrame(all_reviews)
fact_reviews["word_number"] = fact_reviews["word_number"].astype("Int64")

# 3. Exportacao para o Google Drive
OUTPUT_DIR = "/content/drive/MyDrive/TechChallenge_Fase4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

path_products = os.path.join(OUTPUT_DIR, "dim_products.csv")
path_reviews = os.path.join(OUTPUT_DIR, "fact_reviews.csv")

dim_products.to_csv(path_products, index=False, encoding="utf-8-sig", na_rep="")
fact_reviews.to_csv(path_reviews, index=False, encoding="utf-8-sig", na_rep="")

# Validacao rapida do Bloco 4
print(f"Arquivos salvos em: {OUTPUT_DIR}")
print(f"dim_products.csv -> {dim_products.shape[0]} linhas | {dim_products.shape[1]} colunas")
print(f"fact_reviews.csv -> {fact_reviews.shape[0]} linhas | {fact_reviews.shape[1]} colunas")
print("\nNulos em dim_products:")
print(dim_products.isna().sum())
print("\nNulos em fact_reviews:")
print(fact_reviews.isna().sum())
display(dim_products.head(3))
display(fact_reviews.head(3))